In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [ ]:
driver_gsmf = np.loadtxt("/cosmos_storage/home/fgmaion/MTNG-resims/data/driver_GAMA_GSMF.csv", skiprows=3, delimiter=",")
wang_gsmf = np.loadtxt("/cosmos_storage/home/fgmaion/MTNG-resims/data/Wang2024_Table3_SMF_z0-0.2.csv", skiprows=9, delimiter=",")
sdss_gsmf = np.loadtxt("/cosmos_storage/home/fgmaion/MTNG-resims/data/Li_White_2009.csv", skiprows=0, delimiter=",")
bernardi_gsmf = np.loadtxt("/cosmos_storage/home/fgmaion/MTNG-resims/data/Bernardi2018_TableB1B2_GSMF.csv", skiprows=22, delimiter=",")

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.errorbar(driver_gsmf[:,0] + 2*np.log10(0.7), driver_gsmf[:,1] - 3*np.log10(0.7) + 0.0807, yerr=driver_gsmf[:,2], fmt="o", label="GAMA GSMF", color="C0", ms=2)
ax.errorbar(wang_gsmf[:,0], wang_gsmf[:,1], yerr=[wang_gsmf[:,2], wang_gsmf[:,3]], fmt="o", label="DESI GSMF", color="C3", ms=2)
ax.plot(sdss_gsmf[:,0], sdss_gsmf[:,1], label="Li and White (2009)", color="C2")
for i in range(1,5):
    ax.plot(bernardi_gsmf[:,0] + 2*np.log10(0.7), bernardi_gsmf[:,i] - 3*np.log10(0.7), color="C1", marker='^', ms=1, ls='')

ax.set_xlabel('$\log_{10}(M_*/h^2 M_{\odot})$', fontsize=14)
ax.set_ylabel('$\log_{10}(\Phi/[h^3$ Mpc$^{-3}$dex$^{-1}])$', fontsize=14)


ax.legend()

In [ ]:
"""
Two-panel comparison plot:
  Top    : Bernardi+2018 four GSMF variants (Ser/SerExp x M14d/M14df), their
           mean with systematic error bars, and the GAMA/Driver+2022 GSMF.
  Bottom : Residuals of all curves vs the Bernardi mean (in dex).

All curves are converted to h-units (h = 0.7) for consistency with GAMA's
Driver+2022 h70-convention values.

Colorblind-safe palette: Paul Tol "bright" (avoids red/green confusion).
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ---------------------------------------------------------------------------
# Load Bernardi+2018 (Tables B1+B2 combined CSV we built earlier)
# Columns: log_Mstar, Ser_M14d, Ser_M14df, SerExp_M14d, SerExp_M14df
# ---------------------------------------------------------------------------
bernardi_gsmf = np.genfromtxt(
    '/cosmos_storage/home/fgmaion/MTNG-resims/data/Bernardi2018_TableB1B2_GSMF.csv',
    delimiter=',', comments='#', skip_header=1
)

# ---------------------------------------------------------------------------
# Convert Bernardi from physical units (H0=70) to h-units (h=0.7)
#   log M*_h-units = log M*_physical + 2*log10(0.7) = log M* - 0.310
#   log Phi_h-units = log Phi_physical - 3*log10(0.7) = log Phi + 0.464
# Note this is OPPOSITE of converting h-units->physical (which we usually do).
# Here we go physical->h, so signs flip.
#
# Wait - re-checking carefully:
#   M*_phys = M*_h-units * h^-2  =>  log M*_phys = log M*_h-units + 2*log10(h^-1)
#                                                = log M*_h-units - 2*log10(h)
#   So  log M*_h-units = log M*_phys + 2*log10(h) = log M*_phys + 2*log10(0.7)
#                                                  = log M*_phys - 0.310
#   Phi_phys = Phi_h-units * h^3  =>  log Phi_phys = log Phi_h-units + 3*log10(h)
#   So  log Phi_h-units = log Phi_phys - 3*log10(h) = log Phi_phys + 0.464
# ---------------------------------------------------------------------------
dM_h_corr   = 2 * np.log10(0.7/0.6774)    #
dPhi_h_corr = -3 * np.log10(0.7/0.6774)         #

logM_h    = bernardi_gsmf[:, 0] + dM_h_corr       # Mass
logPhi_h  = bernardi_gsmf[:, 1:] + dPhi_h_corr         # Density (4 columns)

# Mean across the 4 variants and max-abs-deviation systematic error
mean_logPhi = np.mean(logPhi_h, axis=1)
diffs       = logPhi_h - mean_logPhi[:, None]
sigma       = np.max(np.abs(diffs), axis=1)               # max distance from mean

# ---------------------------------------------------------------------------
# Load GAMA / Driver+2022 GSMF
# Expecting a CSV with columns: log_Mstar, log_Phi, err_lower, err_upper
# (already in h70-units = h-units convention at h=0.7, so no shift needed
#  to compare with the converted Bernardi above).
#
# Edit this path to point to your GAMA file. If you don't have it as a CSV
# yet, comment out the GAMA block below.
# ---------------------------------------------------------------------------
GAMA_PATH = '/cosmos_storage/home/fgmaion/MTNG-resims/data/driver_GAMA_GSMF.csv'   # <-- edit me
try:
    gama = np.loadtxt(GAMA_PATH, delimiter=',', comments='#')
    gama[:,0] += dM_h_corr   # Shift GAMA mass to physicsal units with MTNG value of h
    gama[:,1] += dPhi_h_corr # Shift GAMA Phi to physical units with MTNG value of h
    have_gama = True
except (OSError, ValueError):
    have_gama = False
    print(f"Note: could not load GAMA from {GAMA_PATH}; will skip the GAMA points.")

# ---------------------------------------------------------------------------
# Colorblind-safe palette (Paul Tol "bright", deuteranopia/protanopia-safe)
# https://personal.sron.nl/~pault/
# Using 4 distinguishable colors for the Bernardi variants + black for mean
# + a distinct color for GAMA.
# ---------------------------------------------------------------------------
tol_bright = {
    'blue':   '#0077BB',
    'cyan':   '#33BBEE',
    'red': '#CC3311',
    'teal': '#009988',
    'magenta': '#EE7733',
}
bernardi_colors = [tol_bright['blue'],   # Ser_M14d
                   tol_bright['teal'],   # Ser_M14df
                   tol_bright['red'], # SerExp_M14d
                   tol_bright['magenta']] # SerExp_M14df
bernardi_markers = ['o', 's', '^', 'D']
bernardi_labels  = ['Sérsic, Dusty',
                    'Sérsic, Dust-free',
                    'SerExp, Dusty',
                    'SerExp, Dust-free']
gama_color  = '#33BBEE'   # rose, distinct from all Bernardi colors and CB-safe
mean_color  = 'k'

# ---------------------------------------------------------------------------
# Figure with two stacked panels sharing x
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(7.0, 6.5), dpi=200)
gs  = GridSpec(2, 1, height_ratios=[3, 1.3], hspace=0.05, figure=fig)
ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

for spine in ax_top.spines.values():
    spine.set_linewidth(2.5)

for spine in ax_bot.spines.values():
    spine.set_linewidth(2.5)


# --- Top panel: GSMFs ------------------------------------------------------
for i in range(4):
    ax_top.plot(logM_h, logPhi_h[:, i],
                linestyle='-', color=bernardi_colors[i], alpha=1,
                label=bernardi_labels[i])

# Bernardi mean with sigma error bars
ax_top.errorbar(logM_h, mean_logPhi, yerr=sigma,
                color=mean_color, marker='o', ms=4,
                linestyle='-', linewidth=1.5, elinewidth=0.7,
                capsize=2, label='Mean ± max systematic')

# GAMA points
if have_gama:
    ax_top.errorbar(gama[:, 0], gama[:, 1]+0.0807,
                    yerr=gama[:, 2],
                    color=gama_color, marker='D', ms=4,
                    linestyle='', elinewidth=0.7, capsize=2,
                    label='GAMA / Driver+22')

ax_top.set_ylim(-7.2,-1.5)
ax_top.set_xlim(8.5, 12.5)
ax_top.set_ylabel(r'$\log_{10}[\,\Phi\,/\,(\mathrm{Mpc}^{-3}\,\mathrm{dex}^{-1})\,]$', fontsize=14)
ax_top.legend(loc='lower left', fontsize=11, ncol=2)

ax_top.minorticks_on()
ax_top.tick_params(which='both', bottom=True, top=True, right=True, direction='in',)  # hide x-ticks on top panel
ax_top.tick_params(axis='both', which='major', width=2.5, length=8, labelsize=12)
ax_top.tick_params(axis='both', which='minor', width=1.5, length=4)
plt.setp(ax_top.get_xticklabels(), visible=False)

# --- Bottom panel: residuals (log10 ratio = dex offset from Bernardi mean) -
for i in range(4):
    ax_bot.plot(logM_h, logPhi_h[:, i] - mean_logPhi,
                linestyle='-', color=bernardi_colors[i], alpha=1)

# Sigma band around 0 for the Bernardi mean
ax_bot.fill_between(logM_h, -sigma, sigma,
                    color=mean_color, alpha=0.15,
                    label=r'Max-systematic band', edgecolor='none')
ax_bot.axhline(0.0, color=mean_color, lw=1.0)

if have_gama:
    # Interpolate Bernardi mean onto GAMA mass grid for residual
    bern_at_gama = np.interp(gama[:, 0], logM_h, mean_logPhi,
                             left=np.nan, right=np.nan)
    ax_bot.errorbar(gama[:, 0], gama[:, 1] - bern_at_gama,
                    yerr=gama[:, 2],
                    color=gama_color, marker='D', ms=4,
                    linestyle='', elinewidth=0.7, capsize=2)

ax_bot.set_xlabel(r'$\log_{10}[\,M_*\,/\,M_\odot\,]$', fontsize=14)
ax_bot.set_ylabel(r'$\Delta\log_{10}\Phi$', fontsize=14)

ax_bot.minorticks_on()
ax_bot.set_xlim(8.5, 12.5)
ax_bot.tick_params(which='both', bottom=True, top=True, right=True, direction='in')  # hide x-ticks on top panel
ax_bot.tick_params(axis='both', which='major', width=2.5, length=8, labelsize=12)
ax_bot.tick_params(axis='both', which='minor', width=1.5, length=4)
ax_bot.legend(loc='lower left', fontsize=10, framealpha=1)

# Reasonable y-limits for residual panel
ax_bot.set_ylim(-0.75, 0.75)

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# Create an inset axis inside ax_top
ax_inset = inset_axes(ax_top, width="30%", height="30%", loc='center')

for spine in ax_inset.spines.values():
    spine.set_linewidth(2.5)

# Plot the same data on the inset
for i in range(4):
    ax_inset.plot(logM_h, logPhi_h[:, i],
                linestyle='-', color=bernardi_colors[i], alpha=1)

# Set limits to zoom in on the low-mass tail
ax_inset.set_xlim(9, 10.5)
ax_inset.set_ylim(-2.5, -1.75)
ax_inset.minorticks_on()
ax_inset.tick_params(which='both', bottom=True, top=True, right=True, direction='in')  # hide x-ticks on top panel
ax_inset.tick_params(axis='both', which='major', width=2.5, length=8, labelsize=12)
ax_inset.tick_params(axis='both', which='minor', width=1.5, length=4)
if have_gama:
    ax_inset.errorbar(gama[:, 0], gama[:, 1]+0.0807,
                    yerr=gama[:, 2],
                    color=gama_color, marker='D', ms=4,
                    linestyle='', elinewidth=0.7, capsize=2)


# Optionally, mark the region on the main plot
mark_inset(ax_top, ax_inset, loc1=2, loc2=4, fc="none", ec="0.5")

out = '/cosmos_storage/home/fgmaion/MTNG-resims/results/images/Bernardi_vs_GAMA_GSMF.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')

In [ ]:
gsmf_stitch = np.loadtxt('/cosmos_storage/home/fgmaion/MTNG-resims/data/GAMA_SDSS_stitched_GSMF_h0p6774.csv',
                  delimiter=',', comments='#', skiprows=33,
                  usecols=(0, 1, 2))

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.errorbar(gsmf_stitch[:,0], gsmf_stitch[:,1], yerr=gsmf_stitch[:,2], label="GAMA/SDSS stitched GSMF", color="k", marker='o', capsize=5)

In [ ]:
gsmf_z = np.loadtxt('/cosmos_storage/home/fgmaion/MTNG-resims/data/moustakas.csv',
                  delimiter=',', skiprows=1)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

z_list = np.unique(gsmf_z[:,0])

for i in range(len(z_list)):
    sel_z = gsmf_z[:,0] == z_list[i]
    ax.errorbar(gsmf_z[sel_z,2], gsmf_z[sel_z,3], yerr=(gsmf_z[sel_z,4], gsmf_z[sel_z,5]),\
        label=f"Moustakas+2013 GSMF at z={z_list[i]:.1f}", marker='o', capsize=5)


In [ ]:
gama_interp = scipy.interpolate.interp1d(driver_gsmf[:,0], driver_gsmf[:,1], bounds_error=False, fill_value=np.nan)
bernardi_interp = scipy.interpolate.interp1d(bernardi_gsmf[:,0], np.mean(bernardi_gsmf[:,1:], axis=1), bounds_error=False, fill_value=np.nan)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')

mstar = np.logspace(9.5,11.2,100)
ax.plot(mstar, gama_interp(np.log10(mstar)), label="GAMA GSMF", color="C0")
ax.plot(mstar, bernardi_interp(np.log10(mstar)), label="Bernardi GSMF", color="C1")